In [1]:
import pandas as pd
import os

In [2]:
# INSTALL PyTest to test logic
%pip install pytest
%pip install ipytest

import ipytest

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
ipytest.autoconfig()

In [4]:
# CODE FROM sap_audit_logic

def generate_lock_recommendations(df, inactivity_threshold=90, today=None):
    """
    Takes a DataFrame (from SAP system) and returns users requiring review.
    """

    # Check if variable "today" already has a date assigned
    # is keyword in Python: https://www.w3schools.com/python/ref_keyword_is.asp
    # is tests the IDENTITY of an object / variable. It does NOT test the equality of two variables.
    if today is None:
        today = datetime.today()

    # print(f"today is: {today}")
    

    df = df.copy() # saves copy of originally passed in DataFrame to locally scoped variable df

    # Convert LastLogon to proper date/time format
    df["LastLogon"] = pd.to_datetime(df["LastLogon"])

    # Calculate inactivity
    # today = datetime.today() # saves today's date in variable "today"
    df["DaysInactive"] = (today - df["LastLogon"]).dt.days # Adds new column "DaysInactive" to df showing number of days each user has been inactive

    # Remove technical users
    # Checks / updates DF at index position represented by "UserType" column
    # [~df["UserType"].isin(["System", "Communication"]) - Ensures that user types "System" and "Communication" are NOT shown in UserType column 
    # result of operation on ~df["UserType"].isin(["System", "Communication"]) stored to variable df and modifies "UserType" column ONLY!
    df = df[~df["UserType"].isin(["System", "Communication"])]

    # Remove already locked users
    # Checks / updates DF at index position represented by "Locked" column
    # to ensure that any users that are NOT locked (["Locked"] == "No") are removed from column "Locked"
    # result of operation on df["Locked"] == "No" is stored to variable df and modifies ONLY "Locked" column ONLY!
    df = df[df["Locked"] == "No"]

    # Flag inactivity
    # Checks DF at index position/ column "DaysInactive" for any users w/ value greater than 90 (see default parameter of function)
    # result of operation df["DaysInactive"] > inactivity_threshold are added to existing df in new column "InactiveFlag"
    df["InactiveFlag"] = df["DaysInactive"] > inactivity_threshold

    # Flag high-risk roles
    # Checks DF at index position/ column "Role" for any strings containing (.str.contains) words ""SAP_ALL"" or "ADMIN" disregarding case-sensitivity + n/a values
    # result of operation df["Role"].str.contains("SAP_ALL|ADMIN", case=False, na=False) added to existing df in new column "HighRiskRole"
    df["HighRiskRole"] = df["Role"].str.contains("SAP_ALL|ADMIN", case=False, na=False)

    # Filter review population
    # Checks DF at index positions/ columns "InactiveFlag" and "HighRiskRole" for value True
    # In other words if true a user has an inactive flag (inactive for more than 90 days) OR (|) if true a user has role containing words "sap_all" or "admin"
    # result of operation df[(df["InactiveFlag"]) | (df["HighRiskRole"])] to be written into NEW DF named "review_df"
    review_df = df[(df["InactiveFlag"]) | (df["HighRiskRole"])]
    
    return review_df

In [20]:
%%ipytest
# THIS CODE WILL BE PLACED INTO SEPARATE FILE NAMED test_sap_audit_logic.py

import pandas as pd
# from sap_audit_logic import generate_lock_recommendations
from datetime import datetime, timedelta

# def build_test_df():
#     # today = datetime.today()

#     data = [
#         # Active dialog user (should NOT appear)
#         ["USER1", "Dialog", today - timedelta(days=10), "No", "Z_AP_CLERK"],

#         # Inactive dialog user > 90 days (should appear)
#         ["USER2", "Dialog", today - timedelta(days=120), "No", "Z_AR_CLERK"],

#         # Exactly 90 days (should NOT appear)
#         ["USER3", "Dialog", today - timedelta(days=90), "No", "Z_AR_CLERK"],

#         # Locked inactive user (should NOT appear)
#         ["USER4", "Dialog", today - timedelta(days=200), "Yes", "Z_MM_USER"],

#         # System user inactive (should NOT appear)
#         ["USER5", "System", today - timedelta(days=200), "No", "Z_BATCH"],

#         # High-risk SAP_ALL (should appear)
#         ["USER6", "Dialog", today - timedelta(days=5), "No", "SAP_ALL"],

#         # High-risk ADMIN role (should appear)
#         ["USER7", "Dialog", today - timedelta(days=5), "No", "Z_ADMIN_ROLE"],
#     ]

#     df = pd.DataFrame(data, columns=["UserID", "UserType", "LastLogon", "Locked", "Role"])
#     return df

# READ DATA FROM CSV
def build_dataFrame():
    df = pd.read_csv("sap_user_logon_export.csv") # TESTING PURPOSES ONLY!
    return df


def test_inactive_users_flagged():
    # build dataframe to be tested
    df = build_dataFrame()

    # variable "result" stores returned df (review_df) from generate_lock_recommentations(df) function
    result = generate_lock_recommendations(df)

    #test assertions
    # assert "USER2" in result["UserID"].values DO NOT DELETE

    # MODIFY ASSERTION FOR "USER2" SUCH THAT TEST PASSES
    # IF TEST PASSES THEN WE KNOW CERTAINLY THE ISSUE IS BECAUSE THE USERNAME "USER2" WAS NOT FOUND 
    # IN LIST OF VALUES FOR RESULT AT INDEX POSITION "UserID"
    # assert "USER2" not in result["UserID"].values # TESTING - DO NOT DELETE!
    # assert "USER3" not in result["UserID"].values

    assert "RC1009" in result ["UserID"].values # THIS SHOULD PASS!


def test_locked_users_removed():
    df = build_dataFrame()
    result = generate_lock_recommendations(df)

    # assert "USER4" not in result["UserID"].values
    pass


def test_system_users_removed():
    df = build_dataFrame()
    result = generate_lock_recommendations(df)

    # assert "USER5" not in result["UserID"].values
    pass


def test_high_risk_roles_flagged(): 
    # build dataframe to be tested
    df = build_dataFrame()
    
    # variable "result" stores returned df (review_df) from generate_lock_recommentations(df) function
    result = generate_lock_recommendations(df)
    
    # test assertions
    # assert "USER6" in result["UserID"].values
    # assert "USER7" in result["UserID"].values
    assert result['Role'].str.contains("SAP_ALL|ADMIN", case=False, na=False).any()


....                                                                                         [100%]
4 passed in 0.25s
